In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

token_ids = torch.tensor([0, 1, 1, 2, 3], dtype = torch.long)
target_ids = torch.tensor([1,1,2,3], dtype= torch.long)
loss_history = []

class ToyAttentionModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.E = nn.Parameter(
            torch.tensor([[1.0, 0.0, 1.0, 0.0],
                          [0.0, 1.0, 0.0, 1.0],
                          [1.0, 1.0, 0.0, 0.0],
                          [0.0, 0.0, 1.0, 1.0]])
        )
        self.W_Q = nn.Parameter(torch.tensor([[1.0,0.0],
                   [0.0,1.0],
                   [1.0,0.0],
                   [0.0,1.0]]))

        self.W_K = nn.Parameter(torch.tensor([[1.0,0.0],
                   [0.0,1.0],
                   [0.0,1.0],
                   [1.0,0.0]]))
        self.W_V = nn.Parameter(torch.tensor([[1.0,0.0,0.0],
                    [0.0,1.0,0.0],
                    [0.0,0.0,1.0],
                    [1.0,1.0,0.0]])) 
        self.W_vocab = nn.Parameter( torch.tensor([[1.0,0.0,-1.0,0.0],
                         [0.0,1.0,1.0,-1.0],
                         [-1.0,0.0,0.0,1.0]]))
        
        
        
    def forward(self, token_ids):        
    
        

        X = self.E[token_ids]
        Q = X @ self.W_Q
        K = X @ self.W_K
        V = X @ self.W_V
        d_k = Q.shape[1]

        S = (Q @ K.T)/(d_k)**0.5

        number_of_tokens = token_ids.shape[0]
        M = torch.full((number_of_tokens, number_of_tokens), float("-inf"), dtype = X.dtype, device = X.device)
        M = torch.triu(M, diagonal = 1)

        S_masked = S + M
        A = torch.softmax(S_masked, dim = 1)
        O = A @ V
        O_train = O[:-1]
        Z = O_train @ self.W_vocab
        return Z

model = ToyAttentionModel()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.1)

for steps in range(1000):

    optimizer.zero_grad() 

    logits = model(token_ids)
    loss = F.cross_entropy(logits, target_ids)

    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if steps % 100 == 0:
        print(steps, loss.item())



print(logits.shape)


print(model.E)
print(model.E.requires_grad)

for name, parameter in model.named_parameters():
    print(name, parameter.shape)



0 1.769589900970459
100 0.6327252388000488
200 0.20708896219730377
300 0.10069804638624191
400 0.03496425226330757
500 0.014992683194577694
600 0.008655178360641003
700 0.005819487385451794
800 0.00427801301702857
900 0.00333156599663198
torch.Size([4, 4])
Parameter containing:
tensor([[ 1.3270,  0.1769,  2.0602,  0.9910],
        [-1.5010,  1.6400, -0.3073,  0.0840],
        [ 2.2841,  0.1439,  0.3631, -1.0485],
        [ 0.0000,  0.0000,  1.0000,  1.0000]], requires_grad=True)
True
E torch.Size([4, 4])
W_Q torch.Size([4, 2])
W_K torch.Size([4, 2])
W_V torch.Size([4, 3])
W_vocab torch.Size([3, 4])
